# LoRA fine-tune best VLM (optional)

This notebook fine-tunes **one** VLM using LoRA on Kvasir-VQA x1.
It trains **two variants** to match the paper’s robustness setting:
- **original images**
- **transformed images**

Outputs:
- `results/<model>_lora_original/predictions.jsonl`
- `results/<model>_lora_transformed/predictions.jsonl`
- `metrics.json` for each run

Heavy artifacts (adapters + processor) are saved under `out/`.

Notes:
- Requires GPU + `accelerate` + `peft`.
- Adjust batch size and max length for your GPU memory.


In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
import sys
import json
from datetime import datetime, timezone
from pathlib import Path
from importlib import metadata as importlib_metadata
%pip install bitsandbytes peft

import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig
try:
    from transformers import AutoModelForImageTextToText
except Exception:
    AutoModelForImageTextToText = None

try:
    from transformers import AutoModelForVision2Seq
except Exception:
    AutoModelForVision2Seq = None

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None

try:
    BITSANDBYTES_VERSION = importlib_metadata.version("bitsandbytes")
    BITSANDBYTES_AVAILABLE = True
except importlib_metadata.PackageNotFoundError:
    BITSANDBYTES_VERSION = None
    BITSANDBYTES_AVAILABLE = False

from peft import LoraConfig, get_peft_model


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Resolve dataset root

def find_kvasir_x1_root() -> Path:
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").exists():
            return (p / "Prototyping_reformat" / "DatasetAnalysis" / "Kvasir_VQA_x1").resolve()
        if (p / "0_dataset_prep").exists() and (p / "1_dataset_analysis").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not locate Kvasir_VQA_x1 root. Set KVASIR_VQA_X1_ROOT.")

ROOT = find_kvasir_x1_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.metrics import normalize_answer, compute_metrics

MANIFEST = ROOT / "0_dataset_prep" / "out" / "manifest_x1.parquet"
RESULTS_BASE = ROOT / "2_modeling" / "11_lora_finetune" / "results"
OUT_BASE = ROOT / "2_modeling" / "11_lora_finetune" / "out"

RESULTS_BASE.mkdir(parents=True, exist_ok=True)
OUT_BASE.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RESULTS_BASE:", RESULTS_BASE)
print("OUT_BASE:", OUT_BASE)


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
RESULTS_BASE: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/11_lora_finetune/results
OUT_BASE: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/11_lora_finetune/out


In [4]:
# Config
MODEL_ID = os.getenv("LORA_MODEL_ID", "google/medgemma-4b-it")
MODEL_NAME = os.getenv("LORA_MODEL_NAME", "medgemma")  # used in output folder names

TRAIN_SPLIT = "train"
EVAL_SPLIT = "test"

RUN_VARIANTS = [v.strip().lower() for v in os.getenv("LORA_RUN_VARIANTS", "original").split(",") if v.strip()]

# Enable only when explicitly requested; this preset is too aggressive for many cloud GPUs.
USE_GB200_SPEED_PRESET = os.getenv("LORA_USE_GB200_SPEED_PRESET", "0") == "1"
FAST_DEV_EVAL = False  # set True to cap eval rows during iteration

BATCH_SIZE = int(os.getenv("LORA_BATCH_SIZE", "4"))
EVAL_BATCH_SIZE = int(os.getenv("LORA_EVAL_BATCH_SIZE", str(BATCH_SIZE)))
GRAD_ACCUM_STEPS = int(os.getenv("LORA_GRAD_ACCUM", "4"))
EPOCHS = int(os.getenv("LORA_EPOCHS", "1"))
LR = float(os.getenv("LORA_LR", "2e-4"))
MAX_LENGTH = int(os.getenv("LORA_MAX_LENGTH", "1024"))

MAX_TRAIN_SAMPLES = int(os.getenv("LORA_MAX_TRAIN_SAMPLES", "0")) or None
MAX_EVAL_SAMPLES = int(os.getenv("LORA_MAX_EVAL_SAMPLES", "0")) or None

NUM_WORKERS = int(os.getenv("LORA_NUM_WORKERS", str(min(8, CPU_THREADS))))
PIN_MEMORY = os.getenv("LORA_PIN_MEMORY", "1") == "1"
PERSISTENT_WORKERS = os.getenv("LORA_PERSISTENT_WORKERS", "1") == "1"
PREFETCH_FACTOR = int(os.getenv("LORA_PREFETCH_FACTOR", "2"))

# On GB200, bf16 full-precision LoRA is usually faster than 4-bit quantized loading.
USE_4BIT = os.getenv("USE_4BIT", "0") == "1"
USE_8BIT = os.getenv("USE_8BIT", "0") == "1"
USE_BF16 = os.getenv("LORA_USE_BF16", "1") == "1"

ATTN_IMPL = os.getenv("LORA_ATTN_IMPL", "flash_attention_2")
USE_FAST_PROCESSOR = os.getenv("LORA_USE_FAST_PROCESSOR", "0") == "1"

GEN_KWARGS = {
    "max_new_tokens": int(os.getenv("LORA_MAX_NEW_TOKENS", "64")),
    "do_sample": os.getenv("LORA_DO_SAMPLE", "0") == "1",
}

if USE_GB200_SPEED_PRESET:
    BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16
    GRAD_ACCUM_STEPS = 4
    MAX_LENGTH = 1024

    NUM_WORKERS = min(16, CPU_THREADS)
    PIN_MEMORY = True
    PERSISTENT_WORKERS = NUM_WORKERS > 0
    PREFETCH_FACTOR = 4

    USE_4BIT = False
    USE_8BIT = False
    USE_BF16 = True
    ATTN_IMPL = "flash_attention_2"

    # Generation is a major eval bottleneck.
    GEN_KWARGS["max_new_tokens"] = 32
    GEN_KWARGS["do_sample"] = False

AUTO_H200_TUNING = os.getenv("LORA_AUTO_H200_TUNING", "1") == "1"
if AUTO_H200_TUNING and torch.cuda.is_available() and "h200" in torch.cuda.get_device_name(0).lower():
    if "LORA_BATCH_SIZE" not in os.environ:
        BATCH_SIZE = max(BATCH_SIZE, 8)
    if "LORA_EVAL_BATCH_SIZE" not in os.environ:
        EVAL_BATCH_SIZE = max(EVAL_BATCH_SIZE, 16)
    if "LORA_NUM_WORKERS" not in os.environ:
        NUM_WORKERS = min(16, CPU_THREADS)
        PERSISTENT_WORKERS = NUM_WORKERS > 0
        PREFETCH_FACTOR = max(PREFETCH_FACTOR, 4)
    print("Detected H200 GPU: applied cloud defaults (override with LORA_* env vars).")

if FAST_DEV_EVAL and MAX_EVAL_SAMPLES is None:
    MAX_EVAL_SAMPLES = 4000

print(
    {
        "MODEL_ID": MODEL_ID,
        "BATCH_SIZE": BATCH_SIZE,
        "EVAL_BATCH_SIZE": EVAL_BATCH_SIZE,
        "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
        "EPOCHS": EPOCHS,
        "MAX_LENGTH": MAX_LENGTH,
        "MAX_NEW_TOKENS": GEN_KWARGS.get("max_new_tokens"),
        "USE_4BIT": USE_4BIT,
        "USE_8BIT": USE_8BIT,
        "USE_BF16": USE_BF16,
        "ATTN_IMPL": ATTN_IMPL,
        "USE_FAST_PROCESSOR": USE_FAST_PROCESSOR,
    }
)


In [5]:
# Load manifest
manifest = pd.read_parquet(MANIFEST)
print("rows:", len(manifest))
print("splits:", manifest["split"].value_counts().to_dict())


rows: 159549
splits: {'train': 143594, 'test': 15955}


In [6]:
# Helpers
STRIP_CHARS = "'\""


def format_prompt(question: str, processor) -> str:
    q = question.strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
        ]
        return processor.apply_chat_template(conversation, add_generation_prompt=True)
    return f"User: <image> Question: {q} Assistant:"


def format_prompt_with_answer(question: str, answer: str, processor) -> str:
    q = question.strip()
    a = str(answer).strip()
    if hasattr(processor, "apply_chat_template"):
        conversation = [
            {"role": "system", "content": "You are a concise medical VQA assistant."},
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": q}]},
            {"role": "assistant", "content": a},
        ]
        return processor.apply_chat_template(conversation, add_generation_prompt=False)
    return f"User: <image> Question: {q} Assistant: {a}"


def postprocess(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()
    for prefix in ["assistant:", "assistant", "answer:"]:
        if t.lower().startswith(prefix):
            t = t[len(prefix):].strip()
    return t


def resolve_image_path(p: str) -> str:
    if p is None:
        return None
    s = str(p)
    path = Path(s)
    if path.exists():
        return str(path)
    if "/Prototyping_reformat/" in s:
        suffix = s.split("/Prototyping_reformat/", 1)[1]
        cand = ROOT.parent / "Prototyping_reformat" / suffix
        if cand.exists():
            return str(cand)
    cand = ROOT / s
    if cand.exists():
        return str(cand)
    cand = ROOT / "0_dataset_prep" / "out" / "images" / "all" / Path(s).name
    if cand.exists():
        return str(cand)
    return str(path)


def format_images_for_processor(images, processor):
    # Gemma3 expects a nested list in batched mode: one image-list per text sample.
    if "gemma3" in type(processor).__name__.lower():
        return [[img] for img in images]
    return images


def is_oom_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return isinstance(exc, torch.OutOfMemoryError) or "out of memory" in msg


def validate_image_paths(df: pd.DataFrame, col: str, split_name: str, max_report: int = 5):
    missing = [p for p in df[col].tolist() if (p is None) or (not Path(p).exists())]
    if missing:
        examples = [str(x) for x in missing[:max_report]]
        raise FileNotFoundError(
            f"{split_name}: {len(missing)} image path(s) are missing after resolution. "
            f"Examples: {examples}"
        )


def run_eval_processor_preflight(df_eval: pd.DataFrame, processor, max_length: int):
    pre_n = min(2, len(df_eval))
    if pre_n == 0:
        return
    batch = df_eval.iloc[:pre_n]
    images = []
    for p in batch["_image_resolved"].tolist():
        with Image.open(p) as img:
            images.append(img.convert("RGB"))
    prompts = batch["_prompt"].tolist()
    _ = processor(
        images=format_images_for_processor(images, processor),
        text=prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )


def save_json(path: Path, payload: dict):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, default=str)


In [7]:
def load_model(model_id: str):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available for LoRA training")

    compute_dtype = torch.bfloat16 if USE_BF16 and torch.cuda.is_bf16_supported() else torch.float16
    if USE_BF16 and compute_dtype != torch.bfloat16:
        print("Warning: bfloat16 requested but not supported on this GPU; using float16.")

    bnb_available = bool(globals().get("BITSANDBYTES_AVAILABLE", False))
    bnb_version = globals().get("BITSANDBYTES_VERSION")

    if (USE_4BIT or USE_8BIT) and (BitsAndBytesConfig is None or not bnb_available):
        print(
            "Warning: bitsandbytes is unavailable in the active kernel; "
            "disabling 4/8-bit quantization. Run `%pip install bitsandbytes` "
            "in this notebook and restart the kernel to enable it."
        )
        use_4bit = False
        use_8bit = False
    else:
        use_4bit = USE_4BIT
        use_8bit = USE_8BIT
        if use_4bit or use_8bit:
            print(f"Using bitsandbytes {bnb_version} for quantized loading")

    quant_config = None
    if torch.cuda.is_available() and (use_4bit or use_8bit) and BitsAndBytesConfig is not None:
        try:
            if use_4bit:
                quant_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_compute_dtype=compute_dtype,
                    bnb_4bit_quant_type="nf4",
                )
            elif use_8bit:
                quant_config = BitsAndBytesConfig(load_in_8bit=True)
        except Exception as e:
            print(f"Warning: quantization init failed ({e}); loading without 4/8-bit quantization.")
            quant_config = None
            use_4bit = False
            use_8bit = False

    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")

    try:
        processor = AutoProcessor.from_pretrained(
            model_id,
            trust_remote_code=True,
            token=hf_token,
            use_fast=USE_FAST_PROCESSOR,
        )
        config = AutoConfig.from_pretrained(model_id, trust_remote_code=True, token=hf_token)
    except Exception as e:
        msg = str(e).lower()
        if "gated repo" in msg or "not in the authorized list" in msg or "access to model" in msg:
            hf_user = "unknown"
            try:
                from huggingface_hub import whoami as hf_whoami
                hf_user = hf_whoami().get("name", "unknown")
            except Exception:
                pass
            raise RuntimeError(
                f"Cannot access {model_id} with HF user '{hf_user}'. Ensure this same account is approved, then run huggingface-cli logout/login in the vqa-rag env and retry."
            ) from e
        raise

    device_map = {"": 0}
    dtype = compute_dtype
    load_kwargs = {
        "device_map": device_map,
        "torch_dtype": dtype,
        "trust_remote_code": True,
        "token": hf_token,
    }
    if quant_config is not None:
        load_kwargs["quantization_config"] = quant_config

    attn_impl = str(globals().get("ATTN_IMPL", os.getenv("LORA_ATTN_IMPL", "flash_attention_2"))).strip()
    if attn_impl and attn_impl.lower() != "none":
        load_kwargs["attn_implementation"] = attn_impl

    def _safe_from_pretrained(loader_cls):
        try:
            return loader_cls.from_pretrained(model_id, **load_kwargs)
        except Exception as e:
            if "attn_implementation" in load_kwargs:
                msg = str(e).lower()
                if "attn_implementation" in msg or "flash" in msg:
                    print("Warning: requested attention implementation failed; retrying with model default attention.")
                    load_kwargs.pop("attn_implementation", None)
                    return loader_cls.from_pretrained(model_id, **load_kwargs)
            raise

    is_vision = getattr(config, "vision_config", None) is not None
    if getattr(config, "model_type", "") in {"qwen2_5_vl", "qwen2_vl", "llava", "idefics2", "idefics3", "fuyu", "blip_2", "git"}:
        is_vision = True

    if is_vision:
        if AutoModelForImageTextToText is not None:
            model = _safe_from_pretrained(AutoModelForImageTextToText)
        elif AutoModelForVision2Seq is not None:
            model = _safe_from_pretrained(AutoModelForVision2Seq)
        else:
            raise RuntimeError("No multimodal auto-model class is available. Upgrade transformers.")
    else:
        model = _safe_from_pretrained(AutoModelForCausalLM)

    model.eval()
    return processor, model


In [8]:
class VQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor, max_length: int = 1024):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row.get("_image_resolved")
        if not img_path:
            img_path = resolve_image_path(row["image_abs_path"] if "image_abs_path" in row else row["image_path"])

        with Image.open(img_path) as img:
            image = img.convert("RGB")

        prompt = row.get("_prompt")
        if not prompt:
            prompt = format_prompt_with_answer(row["question"], row["answer"], self.processor)

        effective_max_length = self.max_length
        if "gemma3" in type(self.processor).__name__.lower():
            # Gemma3 expands image placeholders into many tokens.
            effective_max_length = max(effective_max_length, 1024)

        try:
            inputs = self.processor(
                images=image,
                text=prompt,
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=effective_max_length,
            )
        except ValueError as e:
            if "Mismatch in" not in str(e) or "image token count" not in str(e):
                raise
            raise ValueError("Input got truncated before image tokens completed. Increase LORA_MAX_LENGTH (try 1024 or 1536).") from e

        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs.get("attention_mask", None)
        if attention_mask is not None:
            attention_mask = attention_mask.squeeze(0)

        labels = input_ids.clone()
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        item = {
            "input_ids": input_ids,
            "labels": labels,
        }
        if attention_mask is not None:
            item["attention_mask"] = attention_mask
        if "pixel_values" in inputs:
            item["pixel_values"] = inputs["pixel_values"].squeeze(0)
        return item


In [9]:
class VQACollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = 0 if pad_token_id is None else int(pad_token_id)

    def __call__(self, batch):
        out = {
            "input_ids": torch.nn.utils.rnn.pad_sequence(
                [b["input_ids"] for b in batch],
                batch_first=True,
                padding_value=self.pad_token_id,
            ),
            "labels": torch.nn.utils.rnn.pad_sequence(
                [b["labels"] for b in batch],
                batch_first=True,
                padding_value=-100,
            ),
        }

        if "attention_mask" in batch[0]:
            out["attention_mask"] = torch.nn.utils.rnn.pad_sequence(
                [b["attention_mask"] for b in batch],
                batch_first=True,
                padding_value=0,
            )
        if "pixel_values" in batch[0]:
            out["pixel_values"] = torch.stack([b["pixel_values"] for b in batch])

        return out


In [10]:
def train_lora(df_train: pd.DataFrame, df_eval: pd.DataFrame, variant_name: str):
    results_dir = RESULTS_BASE / f"{MODEL_NAME}_lora_{variant_name}"
    results_dir.mkdir(parents=True, exist_ok=True)

    out_dir = OUT_BASE / f"{MODEL_NAME}_lora_{variant_name}"
    out_dir.mkdir(parents=True, exist_ok=True)

    if len(df_train) == 0 or len(df_eval) == 0:
        print(f"[skip] {variant_name}: empty split (train={len(df_train)}, eval={len(df_eval)})")
        return

    if MAX_TRAIN_SAMPLES is not None and len(df_train) > MAX_TRAIN_SAMPLES:
        df_train = df_train.sample(MAX_TRAIN_SAMPLES, random_state=42).reset_index(drop=True)
        print(f"[{variant_name}] using subset train rows: {len(df_train)}")
    if MAX_EVAL_SAMPLES is not None and len(df_eval) > MAX_EVAL_SAMPLES:
        df_eval = df_eval.sample(MAX_EVAL_SAMPLES, random_state=42).reset_index(drop=True)
        print(f"[{variant_name}] using subset eval rows: {len(df_eval)}")

    processor, model = load_model(MODEL_ID)

    train_image_col = "image_abs_path" if "image_abs_path" in df_train.columns else "image_path"
    eval_image_col = "image_abs_path" if "image_abs_path" in df_eval.columns else "image_path"

    df_train = df_train.copy()
    df_eval = df_eval.copy()

    # Precompute resolved image paths/prompts once to avoid repeated CPU work every batch.
    df_train["_image_resolved"] = df_train[train_image_col].map(resolve_image_path)
    df_eval["_image_resolved"] = df_eval[eval_image_col].map(resolve_image_path)

    df_train["_prompt"] = [
        format_prompt_with_answer(q, a, processor)
        for q, a in zip(df_train["question"].tolist(), df_train["answer"].tolist())
    ]
    df_eval["_prompt"] = [format_prompt(q, processor) for q in df_eval["question"].tolist()]

    validate_image_paths(df_train, "_image_resolved", f"{variant_name}/train")
    validate_image_paths(df_eval, "_image_resolved", f"{variant_name}/eval")
    run_eval_processor_preflight(df_eval, processor, max_length=MAX_LENGTH)

    run_config = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "variant": variant_name,
        "model_id": MODEL_ID,
        "model_name": MODEL_NAME,
        "train_rows": int(len(df_train)),
        "eval_rows": int(len(df_eval)),
        "batch_size": int(BATCH_SIZE),
        "eval_batch_size": int(EVAL_BATCH_SIZE),
        "grad_accum": int(GRAD_ACCUM_STEPS),
        "epochs": int(EPOCHS),
        "lr": float(LR),
        "max_length": int(MAX_LENGTH),
        "max_new_tokens": int(GEN_KWARGS.get("max_new_tokens", 0)),
        "use_4bit": bool(USE_4BIT),
        "use_8bit": bool(USE_8BIT),
        "use_bf16": bool(USE_BF16),
        "attn_impl": str(ATTN_IMPL),
        "use_fast_processor": bool(USE_FAST_PROCESSOR),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }
    save_json(results_dir / "run_config.json", run_config)

    target_modules_env = os.getenv("LORA_TARGET_MODULES", "")
    if target_modules_env:
        target_modules = [m.strip() for m in target_modules_env.split(",") if m.strip()]
    else:
        candidate_modules = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
        target_modules = sorted({name.rsplit(".", 1)[-1] for name, _ in model.named_modules() if name.rsplit(".", 1)[-1] in candidate_modules})
    if not target_modules:
        raise RuntimeError("Could not infer LoRA target modules; set LORA_TARGET_MODULES env var.")
    print("LoRA target modules:", target_modules)

    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
    )
    model = get_peft_model(model, lora_cfg)
    if hasattr(model, "config") and hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    model.train()

    num_workers = max(0, NUM_WORKERS)
    loader_kwargs = {}
    if num_workers > 0:
        loader_kwargs["num_workers"] = num_workers
        loader_kwargs["persistent_workers"] = PERSISTENT_WORKERS
        loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR
    if PIN_MEMORY:
        loader_kwargs["pin_memory"] = True

    train_ds = VQADataset(df_train, processor, max_length=MAX_LENGTH)
    collate_fn = VQACollator(processor.tokenizer.pad_token_id)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        **loader_kwargs,
    )

    optimizer = AdamW(model.parameters(), lr=LR)
    epoch_losses = []

    for epoch in range(EPOCHS):
        total_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f"train {variant_name} e{epoch+1}")):
            batch = {k: v.to(model.device, non_blocking=PIN_MEMORY) for k, v in batch.items()}
            try:
                out = model(**batch)
            except Exception as e:
                if is_oom_error(e):
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    raise RuntimeError(
                        "CUDA OOM during training. Reduce LORA_BATCH_SIZE (e.g., 2 or 1) "
                        "or increase LORA_GRAD_ACCUM."
                    ) from e
                raise
            loss = out.loss / GRAD_ACCUM_STEPS
            loss.backward()
            total_loss += loss.item()

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()

        if len(train_loader) % GRAD_ACCUM_STEPS != 0:
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = total_loss / max(1, len(train_loader))
        epoch_losses.append(float(avg_loss))
        print(f"epoch {epoch+1} avg loss: {avg_loss:.4f}")
        save_json(
            results_dir / "train_progress.json",
            {
                "variant": variant_name,
                "epoch_losses": epoch_losses,
                "completed_epochs": int(epoch + 1),
                "total_epochs": int(EPOCHS),
            },
        )

    model.save_pretrained(out_dir / "lora_adapters")
    processor.save_pretrained(out_dir / "processor")

    if hasattr(model, "config") and hasattr(model.config, "use_cache"):
        model.config.use_cache = True
    model.eval()

    preds = []
    eval_bs = max(1, EVAL_BATCH_SIZE)
    start = 0
    pred_partial_path = results_dir / "predictions.partial.jsonl"
    if pred_partial_path.exists():
        pred_partial_path.unlink()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    pbar = tqdm(total=len(df_eval), desc=f"eval {variant_name}")
    try:
        while start < len(df_eval):
            bs = min(eval_bs, len(df_eval) - start)
            batch = df_eval.iloc[start : start + bs]
            images = []
            for p in batch["_image_resolved"].tolist():
                with Image.open(p) as img:
                    images.append(img.convert("RGB"))

            prompts = batch["_prompt"].tolist()
            images_for_processor = format_images_for_processor(images, processor)

            try:
                inputs = processor(
                    images=images_for_processor,
                    text=prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_LENGTH,
                )
                inputs = {k: v.to(model.device, non_blocking=PIN_MEMORY) for k, v in inputs.items()}
                with torch.inference_mode():
                    out = model.generate(**inputs, **GEN_KWARGS)
            except Exception as e:
                if is_oom_error(e) and bs > 1:
                    next_bs = max(1, bs // 2)
                    print(f"[eval] OOM at batch size {bs}; retrying with batch size {next_bs}")
                    eval_bs = next_bs
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                raise

            prompt_len = inputs["input_ids"].shape[1]
            generated = out[:, prompt_len:] if out.shape[1] > prompt_len else out
            decoded = processor.batch_decode(generated, skip_special_tokens=True)
            batch_preds = [postprocess(t) for t in decoded]
            preds.extend(batch_preds)
            batch_out = batch.drop(columns=["_prompt", "_image_resolved"], errors="ignore").copy()
            batch_out["pred_raw"] = batch_preds
            batch_out["pred_norm"] = batch_out["pred_raw"].apply(normalize_answer)
            with open(pred_partial_path, "a", encoding="utf-8") as pf:
                batch_out.to_json(pf, orient="records", lines=True)
            start += bs
            pbar.update(bs)
    finally:
        pbar.close()

    out_df = pd.read_json(pred_partial_path, orient="records", lines=True)
    if len(out_df) != len(df_eval):
        raise RuntimeError(
            f"Eval output row mismatch: got {len(out_df)} predictions for {len(df_eval)} eval rows."
        )

    pred_path = results_dir / "predictions.jsonl"
    out_df.to_json(pred_path, orient="records", lines=True)
    pred_partial_path.unlink(missing_ok=True)

    metrics = {"overall": compute_metrics(out_df["pred_norm"], out_df["answer"])}
    save_json(results_dir / "metrics.json", metrics)
    save_json(
        results_dir / "artifacts.json",
        {
            "predictions": str(pred_path),
            "metrics": str(results_dir / "metrics.json"),
            "run_config": str(results_dir / "run_config.json"),
            "train_progress": str(results_dir / "train_progress.json"),
            "adapter_dir": str(out_dir / "lora_adapters"),
            "processor_dir": str(out_dir / "processor"),
        },
    )

    print("saved results:", results_dir)
    print("saved heavy artifacts:", out_dir)


In [11]:
# Run variants

variant_flags = {
    "original": False,
    "transformed": True,
}

if not RUN_VARIANTS:
    raise ValueError("No variants requested. Set LORA_RUN_VARIANTS, e.g. original or original,transformed")

for variant in RUN_VARIANTS:
    flag = variant_flags.get(variant)
    if flag is None:
        raise ValueError(f"Unknown variant: {variant}")

    train_df = manifest[(manifest["split"] == TRAIN_SPLIT) & (manifest["is_transformed"] == flag)].copy()
    eval_df = manifest[(manifest["split"] == EVAL_SPLIT) & (manifest["is_transformed"] == flag)].copy()

    print(variant, "train:", len(train_df), "eval:", len(eval_df))
    if len(train_df) == 0 or len(eval_df) == 0:
        print(f"[skip] {variant}: empty split")
        continue
    train_lora(train_df, eval_df, variant)


original train: 143594 eval: 15955
Using bitsandbytes 0.49.1 for quantized loading


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LoRA target modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']


train original e1:   0%|          | 0/17950 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 144.00 MiB. GPU 0 has a total capacity of 23.54 GiB of which 99.81 MiB is free. Including non-PyTorch memory, this process has 22.83 GiB memory in use. Of the allocated memory 22.03 GiB is allocated by PyTorch, and 500.40 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)